# Hong Kong Air Quality Briefing System (Groupby · AIDM7360)

**End-to-end pipeline:** Raw CSV/XML/GeoJSON/JSON ETL → 8-table SQLite database (`air_quality.db`) → 9 parameterized SQL queries (`saved_queries`) → Full 8-table CRUD → Shared 3-format template engine (`Text` / `Markdown` / `HTML`) → Interactive Notebook UI + Matplotlib charts + Folium 18-district map → Verified static website (`docs/`).

### Setup (Local Jupyter & Google Colab)
- **Local**: Open this notebook from the repository root (`AIDM7360_group_project/`), install dependencies with `pip install -r requirements.txt`, and run all cells top-to-bottom.
- **Google Colab**: Mount Google Drive, `%cd` into your uploaded project root (or set `os.environ["AQHI_PROJECT_ROOT"]`), and install `requirements.txt` if needed (`%pip install -r requirements.txt`).
- All write demonstrations (CRUD roundtrips and ad-hoc report persistence) execute on an isolated `TemporaryDirectory` SQLite backup (`demo_db`) so `data/db/air_quality.db` remains untouched.

In [ ]:
import atexit
from contextlib import ExitStack, closing
import json
import os
from pathlib import Path
import sqlite3
import sys
from tempfile import TemporaryDirectory

configured = os.environ.get("AQHI_PROJECT_ROOT")
candidates = [Path(configured)] if configured else [Path.cwd(), *Path.cwd().parents]
project = next(
    (
        p.resolve()
        for p in candidates
        if (p / "src" / "schema.py").is_file() and (p / "data" / "db" / "air_quality.db").is_file()
    ),
    None,
)

if project is None and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    colab_default = Path("/content/drive/MyDrive/AIDM7360_group_project")
    if (colab_default / "src" / "schema.py").is_file():
        project = colab_default.resolve()

if project is None:
    raise FileNotFoundError(
        "Locate the project root containing src/schema.py and data/db/air_quality.db, "
        "or set AQHI_PROJECT_ROOT."
    )

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

import pandas as pd
from IPython.display import HTML, Image, Markdown, display

from src import crud
from src.queries import QUERY_DEFINITIONS, run_saved_query, seed_saved_queries
from src.schema import DEFAULT_DB_PATH, PROJECT_ROOT, get_connection
from src.site_export import export_static_site, validate_site_links
from src.task_d_integration import default_cases, demo_database, make_briefing, query_catalog, select
from src.templates import (
    DEFAULT_TEMPLATES,
    TEMPLATE_NAMES,
    generate_report,
    seed_report_templates,
)
from src.viz import (
    build_district_map,
    plot_category_distribution,
    plot_hourly_aqhi,
    plot_monthly_trend,
    plot_station_comparison,
)

assert PROJECT_ROOT == project, f"Unexpected PROJECT_ROOT: {PROJECT_ROOT}"

# Verify all 8 database tables and key EPD/HKO/C&SD data-quality counts
table_names = [
    "districts",
    "stations",
    "hourly_aqhi",
    "hourly_pollutants",
    "daily_weather",
    "saved_queries",
    "report_templates",
    "generated_reports",
]
table_counts = {
    t: select(f'SELECT COUNT(*) AS n FROM "{t}"')[0]["n"]
    for t in table_names
}
display(pd.DataFrame([table_counts]))

aqhi_quality = select("""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN is_missing = 0 THEN 1 ELSE 0 END) AS valid_rows,
           SUM(is_missing) AS missing_rows,
           SUM(CASE WHEN is_missing = 0 AND is_substitute = 1 THEN 1 ELSE 0 END) AS valid_substitutes,
           SUM(CASE WHEN is_missing = 0 AND raw_value = '10+' AND sort_rank = 11 AND numeric_value IS NULL THEN 1 ELSE 0 END) AS serious_10plus_rows
    FROM hourly_aqhi
""")[0]
print("AQHI quality checks:", aqhi_quality)
print("Note: To rebuild data/db/air_quality.db from raw CSV/XML/GeoJSON files, run src.etl_aqhi and src.etl_extra.")

# Create an isolated temporary database snapshot and scratch directory for safe write demos
if "main_resources" in globals():
    main_resources.close()
main_resources = ExitStack()
atexit.register(main_resources.close)
demo_db = main_resources.enter_context(demo_database())
work = Path(main_resources.enter_context(TemporaryDirectory(prefix="aqhi-main-")))
print("Isolated demo snapshot ready:", demo_db)

## 1. Saved SQL Queries (`Q01`–`Q09`)

Task B stores 9 parameterized SQL queries in `saved_queries` covering `JOIN`, `LEFT JOIN`, subqueries, `COUNT/MIN/MAX/SUM/AVG`, `CASE`, `GROUP_CONCAT`, `GROUP BY`, and `IN`.

In [ ]:
saved_catalog = select(
    "SELECT query_code, name, parameter_names FROM saved_queries ORDER BY query_code",
    db_path=demo_db,
)
display(pd.DataFrame(saved_catalog))

q01_apr01 = run_saved_query("Q01_daily_briefing", {"station_name": "Central/Western", "obs_date": "2025-04-01"}, demo_db)
q01_apr14 = run_saved_query("Q01_daily_briefing", {"station_name": "Central/Western", "obs_date": "2025-04-14"}, demo_db)
print("Q01 (2025-04-01):", {k: q01_apr01[0][k] for k in ("station_name", "valid_hours", "min_aqhi", "max_aqhi", "peak_hours", "avg_pm25")})
print("Q01 (2025-04-14 Serious 10+):", {k: q01_apr14[0][k] for k in ("station_name", "valid_hours", "max_aqhi", "numeric_aqhi_hours", "peak_hours", "valid_pm25_hours")})

q02 = run_saved_query("Q02_category_breakdown", {"station_name": "Central/Western", "obs_date": "2025-04-01"}, demo_db)
display(pd.DataFrame(q02)[["category", "hours", "percentage", "valid_hours", "missing_hours", "substitute_hours"]])

q03 = run_saved_query("Q03_station_comparison", {"station_a": "Central/Western", "station_b": "Central", "obs_date": "2025-04-01"}, demo_db)
display(pd.DataFrame(q03)[["station_name", "station_type", "max_aqhi", "avg_aqhi", "avg_pm25", "valid_pm25_hours"]])

q04 = run_saved_query("Q04_weather_pollution", {"station_name": "Central/Western", "start_date": "2025-03-21", "end_date": "2025-03-23"}, demo_db)
display(pd.DataFrame(q04)[["obs_date", "max_aqhi", "avg_pm25", "mean_temp_c", "mean_rh_pct", "rainfall_mm", "mean_wind_kmh"]])

q05 = run_saved_query("Q05_district_map", {"obs_date": "2025-04-14"}, demo_db)
print("Q05 row counts by row_type:", pd.Series([r["row_type"] for r in q05]).value_counts().to_dict())
print("Districts without General station:", [r["district_name"] for r in q05 if r["row_type"] == "district" and r["general_station_count"] == 0])

q06 = run_saved_query("Q06_monthly_summary", {"station_name": "Central/Western", "month": "2025-03"}, demo_db)
print("Q06 (2025-03):", {k: q06[0][k] for k in ("station_name", "month", "valid_hours", "substitute_hours", "avg_aqhi", "max_aqhi")})

q07 = run_saved_query("Q07_high_risk_bulletin", {"station_name": "Central/Western", "start_date": "2025-04-14", "end_date": "2025-04-14"}, demo_db)
print("Q07 (2025-04-14):", {k: q07[0][k] for k in ("obs_date", "max_aqhi", "high_or_above_hours", "high_risk_hours")})

q08 = run_saved_query("Q08_news_flash", {"obs_date": "2025-04-14"}, demo_db)
print("Q08 (2025-04-14):", {k: q08[0][k] for k in ("obs_date", "max_aqhi", "high_or_above_hours", "peak_stations")})

q09 = run_saved_query("Q09_report_history", {}, demo_db)
display(pd.DataFrame(q09))

## 2. Full 8-Table Parameterized CRUD & Foreign-Key Protection

Demonstrates Create, Read, Update, and Delete across all 8 tables on the isolated `demo_db` snapshot, including derived AQHI flag updates from `raw_value` and `ReferencedRowError` protection before deleting referenced parent records.

In [ ]:
def crud_roundtrip(table, create_fn, get_fn, update_fn, delete_fn, fields, changes):
    row_id = create_fn(db_path=demo_db, **fields)
    created = get_fn(row_id, db_path=demo_db)
    assert created is not None, f"Failed to read newly created {table} row"
    assert update_fn(row_id, db_path=demo_db, **changes)
    updated = get_fn(row_id, db_path=demo_db)
    assert all(updated[k] == v for k, v in changes.items()), f"Update mismatch on {table}"
    assert delete_fn(row_id, db_path=demo_db)
    assert get_fn(row_id, db_path=demo_db) is None
    print(f"[CRUD PASS] {table}: created -> read -> updated -> deleted row_id={row_id}")

station_id = q01_apr01[0]["station_id"]
district_id = select("SELECT district_id FROM stations WHERE station_id = ?", (station_id,), demo_db)[0]["district_id"]
q01_id = select("SELECT query_id FROM saved_queries WHERE query_code = ?", ("Q01_daily_briefing",), demo_db)[0]["query_id"]

crud_roundtrip(
    "districts", crud.create_district, crud.get_district, crud.update_district, crud.delete_district,
    {"district_code": "MAIN_D", "english_name": "Main Demo District", "region": "Kowloon"},
    {"area_sqkm": 3.14},
)
crud_roundtrip(
    "stations", crud.create_station, crud.get_station, crud.update_station, crud.delete_station,
    {"code": "MAIN_S", "name": "Main Demo Station", "station_type": "General", "district_id": district_id},
    {"latitude": 22.30, "longitude": 114.17},
)
crud_roundtrip(
    "hourly_aqhi", crud.create_hourly_aqhi, crud.get_hourly_aqhi, crud.update_hourly_aqhi, crud.delete_hourly_aqhi,
    {"station_id": station_id, "obs_date": "2099-01-01", "obs_hour": 24, "raw_value": "4*"},
    {"raw_value": "10+", "numeric_value": None, "sort_rank": 11, "category": "Serious", "is_substitute": 0},
)
crud_roundtrip(
    "hourly_pollutants", crud.create_hourly_pollutant, crud.get_hourly_pollutant, crud.update_hourly_pollutant, crud.delete_hourly_pollutant,
    {"station_id": station_id, "obs_date": "2099-01-01", "obs_hour": 24, "pm25": 11.2, "pm25_unit": "ug/m3"},
    {"pm25": 14.6},
)
crud_roundtrip(
    "daily_weather", crud.create_daily_weather, crud.get_daily_weather, crud.update_daily_weather, crud.delete_daily_weather,
    {"obs_date": "2099-01-01", "mean_temp_c": 19.5},
    {"rainfall_mm": 2.4},
)
crud_roundtrip(
    "saved_queries", crud.create_saved_query, crud.get_saved_query, crud.update_saved_query, crud.delete_saved_query,
    {"query_code": "MAIN_DEMO_Q", "name": "Main Demo Query", "sql_text": "SELECT 1 AS v", "parameter_names": "[]"},
    {"description": "Updated description"},
)

tpl_fields = {
    "query_id": q01_id,
    "name": "Main Demo Template",
    "scope": "territory",
    "required_variables": json.dumps(["max_aqhi"]),
    "text_template": "AQHI {max_aqhi}",
    "markdown_template": "**AQHI {max_aqhi}**",
    "html_template": "<p>AQHI {max_aqhi}</p>",
}
crud_roundtrip(
    "report_templates", crud.create_report_template, crud.get_report_template, crud.update_report_template, crud.delete_report_template,
    tpl_fields,
    {"version": "1.1"},
)

# Nullable station_id + foreign-key protection demonstration on generated_reports -> report_templates
test_tpl_id = crud.create_report_template(db_path=demo_db, **tpl_fields)
test_rep_id = crud.create_generated_report(
    db_path=demo_db,
    template_id=test_tpl_id,
    station_id=None,
    title="Territory Demo Report",
    parameter_values=json.dumps({"obs_date": "2025-04-14"}),
    reporting_period="2025-04-14",
    text_output="AQHI 10+",
    markdown_output="**AQHI 10+**",
    html_output="<p>AQHI 10+</p>",
)
assert crud.get_generated_report(test_rep_id, db_path=demo_db)["station_id"] is None
assert crud.update_generated_report(test_rep_id, db_path=demo_db, title="Updated Territory Demo Report")

try:
    crud.delete_report_template(test_tpl_id, db_path=demo_db)
except crud.ReferencedRowError as exc:
    print("[FK PROTECTION PASS]", exc)
else:
    raise AssertionError("Expected ReferencedRowError when deleting referenced parent template")

assert crud.delete_generated_report(test_rep_id, db_path=demo_db)
assert crud.delete_report_template(test_tpl_id, db_path=demo_db)
print("[CRUD PASS] generated_reports: created (station_id=NULL) -> updated -> deleted")

## 3. Shared Template Engine (`Text` / `Markdown` / `HTML`) & Boundary Cases

`src.templates.generate_report()` pulls live results from `saved_queries`, validates coverage and required variables, converts `10+` and `NULL` safely, attaches the March 22 calculation note when applicable, renders all three formats, and persists only valid formal briefings.

In [ ]:
showcase_specs = [
    ("Daily Station Briefing", {"station_name": "Central/Western", "obs_date": "2025-04-01"}),
    ("Daily Category Breakdown", {"station_name": "Central/Western", "obs_date": "2025-04-01"}),
    ("Comparison of Two Stations", {"station_a": "Central/Western", "station_b": "Central", "obs_date": "2025-04-01"}),
    ("Weather and Pollution Summary", {"station_name": "Central/Western", "start_date": "2025-03-21", "end_date": "2025-03-23"}),
    ("District Map Report", {"obs_date": "2025-04-14"}),
    ("Monthly Station Summary", {"station_name": "Central/Western", "month": "2025-03"}),
    ("High Risk Episode Bulletin", {"station_name": "Central/Western", "start_date": "2025-04-14", "end_date": "2025-04-14"}),
    ("Air Quality News Flash", {"obs_date": "2025-04-14"}),
]

for tpl_name, tpl_params in showcase_specs:
    rid = generate_report(tpl_name, tpl_params, save=True, db_path=demo_db)
    assert isinstance(rid, int), f"Expected saved report_id for {tpl_name}, got {rid}"
    row = crud.get_generated_report(rid, db_path=demo_db)
    print(f"=== {tpl_name} (report_id={rid}) ===")
    print(row["text_output"])
    print()

# Preview Markdown and HTML outputs for the Daily Station Briefing (2025-04-01)
preview = generate_report(
    "Daily Station Briefing",
    {"station_name": "Central/Western", "obs_date": "2025-04-01"},
    save=False,
    db_path=demo_db,
)
display(Markdown(preview["markdown_output"]))
display(HTML(preview["html_output"]))

# Edge case 1: Zero High-risk hours on a valid day (2025-04-01) -> formal briefing with explicit message
low_day = generate_report(
    "High Risk Episode Bulletin",
    {"station_name": "Central/Western", "start_date": "2025-04-01", "end_date": "2025-04-01"},
    save=False,
    db_path=demo_db,
)
assert low_day["formal"] and "No valid observations reached AQHI 7 or above" in low_day["text_output"]
print("Edge case 1 (valid day, 0 high-risk hours):", low_day["text_output"].splitlines()[2])

# Edge case 2: Out-of-coverage date (2026-01-01) -> non-formal fallback, NOT inserted into generated_reports
before_count = select("SELECT COUNT(*) AS n FROM generated_reports", db_path=demo_db)[0]["n"]
no_data = generate_report(
    "Daily Station Briefing",
    {"station_name": "Central/Western", "obs_date": "2026-01-01"},
    save=True,
    db_path=demo_db,
)
after_count = select("SELECT COUNT(*) AS n FROM generated_reports", db_path=demo_db)[0]["n"]
assert no_data["formal"] is False and no_data["report_id"] is None and before_count == after_count
print("Edge case 2 (no valid data, not saved):", no_data["text_output"])

## 4. Interactive Explorer, Matplotlib Charts & Folium 18-District Map

Select any 2025 station, comparison station, date, month, or date range via the widget controls (or edit `interactive_inputs` directly in the parameter cell below when widgets are disabled). District polygons shade General-station daily maxima while keeping the four districts without a General station (`Kowloon City`, `Wan Chai`, `Wong Tai Sin`, `Yau Tsim Mong`) unshaded grey; Roadside stations appear in a separate toggleable layer.

In [ ]:
from IPython.display import clear_output

stations, dates, months = query_catalog()
station_names = [row["name"] for row in stations]
template_rows = select(
    """SELECT t.name, q.query_code, q.parameter_names
       FROM report_templates t
       JOIN saved_queries q USING(query_id)
       ORDER BY q.query_code""",
    db_path=demo_db,
)
cases = default_cases()
first = cases[0]
query_params = {r["name"]: set(json.loads(r["parameter_names"])) for r in template_rows}
preferred = [
    "station_name", "station_type", "district_name", "row_type", "obs_date",
    "month", "category", "hours", "percentage", "valid_hours", "missing_hours",
    "substitute_hours", "min_aqhi", "max_aqhi", "peak_hours", "high_or_above_hours",
    "avg_pm25", "valid_pm25_hours", "high_risk_hours", "peak_stations",
]


def show_briefing(template, station, other, day, month, start, end):
    params = dict(
        station_name=station, station_a=station, station_b=other,
        obs_date=day, month=month, start_date=start, end_date=end,
    )
    result = make_briefing(template, params, demo_db)
    print(
        f"{result.query_code} · {len(result.rows)} SQL result rows · "
        + ("formal template" if result.formal else "query-data fallback")
    )
    frame = pd.DataFrame(result.rows)
    if not frame.empty:
        display(frame[[name for name in preferred if name in frame.columns]].head(24))
    display(HTML(result.outputs["html_output"]))
    return result


def show_charts(station, other, day, month):
    for index, (plot, args) in enumerate((
        (plot_hourly_aqhi, (station, day)),
        (plot_category_distribution, (station, day)),
        (plot_station_comparison, (station, other, day)),
        (plot_monthly_trend, (station, month)),
    )):
        path = plot(*args, work / f"chart-{index}.png", db_path=demo_db)
        display(Image(filename=str(path), width=950))


def show_district_map(day):
    district_map = build_district_map(day, work / f"districts-{day}.html", db_path=demo_db)
    print(day, district_map.task_d_query_source)
    display(district_map)
    return district_map


try:
    if os.environ.get("AQHI_DISABLE_WIDGETS") == "1":
        raise ImportError("Widgets disabled")
    import ipywidgets as widgets
    if "google.colab" in sys.modules:
        from google.colab import output
        output.enable_custom_widget_manager()
except ImportError:
    widgets = None
    print("Widgets unavailable; edit interactive_inputs in the cell below.")

if widgets is not None:
    names = [(f'{r["name"]} ({r["station_type"]})', r["name"]) for r in stations]
    controls = {
        "template": widgets.Dropdown(options=list(TEMPLATE_NAMES), description="Report"),
        "station": widgets.Dropdown(options=names, value=first["station_name"], description="Station"),
        "other": widgets.Dropdown(options=names, value=first["station_b"], description="Second"),
        "day": widgets.Dropdown(options=dates, value=first["obs_date"], description="Date"),
        "month": widgets.Dropdown(options=months, value=first["month"], description="Month"),
        "start": widgets.Dropdown(options=dates, value=first["obs_date"], description="Range start"),
        "end": widgets.Dropdown(options=dates, value=first["obs_date"], description="Range end"),
    }
    for control in controls.values():
        control.layout.width = "min(100%, 480px)"
        control.style.description_width = "105px"
    briefing_button = widgets.Button(description="Generate briefing", button_style="primary")
    chart_button = widgets.Button(description="Draw four charts")
    map_button = widgets.Button(description="Show District Map", layout=widgets.Layout(width="auto"))
    result_area = widgets.Output()

    def update_report_inputs(change=None):
        required = query_params[controls["template"].value]
        for key, param in (("start", "start_date"), ("end", "end_date")):
            controls[key].layout.display = "" if param in required else "none"
        briefing_button.tooltip = "Report fields: " + ", ".join(sorted(required))

    def on_click(button):
        values = {key: control.value for key, control in controls.items()}
        with result_area:
            clear_output(wait=True)
            try:
                if button is briefing_button:
                    show_briefing(**values)
                elif button is map_button:
                    show_district_map(values["day"])
                else:
                    show_charts(*(values[key] for key in ("station", "other", "day", "month")))
            except ValueError as exc:
                print(exc)

    controls["template"].observe(update_report_inputs, names="value")
    update_report_inputs()
    buttons = (briefing_button, chart_button, map_button)
    for button in buttons:
        button.on_click(on_click)
    display(widgets.VBox([
        *controls.values(),
        widgets.HBox(buttons, layout=widgets.Layout(flex_flow="row wrap")),
        result_area,
    ]))

In [ ]:
# Edit these values to explore any 2025 station, date, or month with or without widgets
interactive_inputs = dict(
    template=TEMPLATE_NAMES[0],
    station=first["station_name"],
    other=first["station_b"],
    day=first["obs_date"],
    month=first["month"],
    start=first["obs_date"],
    end=first["obs_date"],
)
briefing = show_briefing(**interactive_inputs)
show_charts(*(interactive_inputs[k] for k in ("station", "other", "day", "month")))
selected_map = show_district_map(interactive_inputs["day"])

## 5. Static Site Export & Link Verification (`docs/`)

Exports the stored reports, Matplotlib PNGs, Folium district maps, and responsive `index.html` into a temporary staging folder and validates every local file and anchor link. To rebuild the repository's `docs/` folder for GitHub Pages, run `python -m src.site_export` from the project root.

In [ ]:
index = export_static_site(output_dir=work / "site")
site_root = index.parent
errors = validate_site_links(site_root)
assert index.is_file() and (site_root / ".nojekyll").is_file() and not errors, errors

print("Verified static site index:", index)
print("Report HTML files:", len(list((site_root / "reports").glob("*.html"))))
print("Report Markdown files:", len(list((site_root / "reports").glob("*.md"))))
print("Chart PNG files:", len(list((site_root / "assets" / "charts").glob("*.png"))))
print("Map HTML files:", len(list((site_root / "maps").glob("*.html"))))